In [1]:
!pip install transformers datasets accelerate evaluate scikit-learn

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 8.2 MB/s eta 0:00:00


In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [3]:
import numpy as np
from datasets import load_dataset
from sklearn.metrics import accuracy_score, classification_report, f1_score
from transformers import (
    AutoModelForSequenceClassification,
    AutoTokenizer,
    Trainer,
    TrainingArguments,
)

from config import DATASET_NAME, MAX_LENGTH, MODEL_NAME, TRAINED_MODEL_DIR, TRAINING_ARGS

In [5]:
def load_emotion_dataset():
    dataset = load_dataset(DATASET_NAME)

    print(f"Train      : {len(dataset['train']):,} samples")
    print(f"Validation : {len(dataset['validation']):,} samples")
    print(f"Test       : {len(dataset['test']):,} samples")
    print(f"Sample     : {dataset['train'][0]}")

    label_names = dataset["train"].features["label"].names
    print(f"Labels     : {label_names}\n")
    return dataset, label_names

In [6]:
def build_tokenizer(model_name: str):
    return AutoTokenizer.from_pretrained(model_name)


def tokenize_dataset(dataset, tokenizer):
    def _tokenize(examples):
        return tokenizer(
            examples["text"],
            padding="max_length",
            truncation=True,
            max_length=MAX_LENGTH,
        )

    return dataset.map(_tokenize, batched=True, remove_columns=["text"])

In [7]:
def make_compute_metrics():
    def compute_metrics(eval_pred):
        predictions, labels = eval_pred
        predictions = np.argmax(predictions, axis=1)
        return {
            "accuracy": accuracy_score(labels, predictions),
            "f1": f1_score(labels, predictions, average="weighted"),
        }
    return compute_metrics

In [8]:
def train(dataset, tokenizer, label_names):
    num_labels = len(label_names)
    model = AutoModelForSequenceClassification.from_pretrained(
        MODEL_NAME, num_labels=num_labels
    )

    tokenized = tokenize_dataset(dataset, tokenizer)

    training_args = TrainingArguments(**TRAINING_ARGS)

    trainer = Trainer(
        model=model,
        args=training_args,
        train_dataset=tokenized["train"],
        eval_dataset=tokenized["validation"],
        compute_metrics=make_compute_metrics(),
    )

    print("Starting training …")
    trainer.train()
    return trainer, tokenized

In [9]:
def evaluate(trainer, tokenized_test, label_names):
    results = trainer.evaluate(tokenized_test)
    print(f"\nTest Accuracy : {results['eval_accuracy']:.4f}")
    print(f"Test F1       : {results['eval_f1']:.4f}")

    predictions = trainer.predict(tokenized_test)
    pred_labels = np.argmax(predictions.predictions, axis=1)
    true_labels = tokenized_test["label"]

    unique_labels = sorted(set(true_labels))
    present_names = [label_names[i] for i in unique_labels]

    print("\nClassification Report:")
    print(
        classification_report(
            true_labels,
            pred_labels,
            labels=unique_labels,
            target_names=present_names,
        )
    )

In [10]:
def save(trainer, tokenizer):
    trainer.save_model(TRAINED_MODEL_DIR)
    tokenizer.save_pretrained(TRAINED_MODEL_DIR)
    print(f"\nModel saved to: {TRAINED_MODEL_DIR}")

In [11]:
if __name__ == "__main__":
    dataset, label_names = load_emotion_dataset()
    tokenizer = build_tokenizer(MODEL_NAME)
    trainer, tokenized = train(dataset, tokenizer, label_names)
    evaluate(trainer, tokenized["test"], label_names)
    save(trainer, tokenizer)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

split/train-00000-of-00001.parquet:   0%|          | 0.00/1.03M [00:00<?, ?B/s]

split/validation-00000-of-00001.parquet:   0%|          | 0.00/127k [00:00<?, ?B/s]

split/test-00000-of-00001.parquet:   0%|          | 0.00/129k [00:00<?, ?B/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.weight', 'distilbert.embeddings.LayerNorm.bias'].
There were unexpected keys in the checkpoint model loaded: ['distilbert.embeddings.LayerNorm.beta', 'distilbert.embeddings.LayerNorm.gamma'].



Test Accuracy : 0.9250
Test F1       : 0.9246

Classification Report:
              precision    recall  f1-score   support

     sadness       0.97      0.96      0.96       581
         joy       0.93      0.97      0.95       695
        love       0.91      0.74      0.81       159
       anger       0.91      0.94      0.92       275
        fear       0.91      0.85      0.88       224
    surprise       0.68      0.85      0.76        66

    accuracy                           0.93      2000
   macro avg       0.89      0.88      0.88      2000
weighted avg       0.93      0.93      0.92      2000



Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]


Model saved to: ./trained_model


In [12]:
!cp -r ./trained_model /content/drive/MyDrive/trained_model

In [16]:
import torch
from transformers import AutoModelForSequenceClassification, AutoTokenizer
from ipywidgets import widgets
from IPython.display import display
from google.colab import drive

drive.mount('/content/drive')

tokenizer = AutoTokenizer.from_pretrained("/content/drive/MyDrive/trained_model")
model = AutoModelForSequenceClassification.from_pretrained("/content/drive/MyDrive/trained_model")
model.eval()

label_names = ["sadness", "joy", "love", "anger", "fear", "surprise"]

EMOTION_RESPONSES = {
    "sadness"  : "I'm sorry you're feeling down. 😔",
    "joy"      : "That sounds wonderful! 😊",
    "love"     : "That's so sweet! ❤️",
    "anger"    : "I can understand your frustration. 😤",
    "fear"     : "That sounds concerning. 🤗",
    "surprise" : "Wow, that's quite unexpected! 😲",
}

def predict_emotion(text):
    inputs = tokenizer(text, return_tensors="pt", truncation=True, padding=True)
    with torch.no_grad():
        outputs = model(**inputs)
    probs = torch.softmax(outputs.logits, dim=1)[0]
    idx = torch.argmax(probs).item()
    return label_names[idx], probs[idx].item()

text_input = widgets.Text(placeholder="Type your message here...", layout=widgets.Layout(width="500px"))
out = widgets.Output()
send_button = widgets.Button(description="Send", button_style="primary")
exit_button = widgets.Button(description="Exit", button_style="danger")

def on_send(b):
    with out:
        text = text_input.value.strip()
        if not text:
            return
        emotion, confidence = predict_emotion(text)
        print(f"You     : {text}")
        print(f"Emotion : {emotion.upper()} ({confidence:.1%})")
        print(f"Bot     : {EMOTION_RESPONSES[emotion]}")
        print("-" * 50)
        text_input.value = ""

def on_exit(b):
    with out:
        print("Bot: Goodbye! 👋")
    text_input.disabled = True
    send_button.disabled = True
    exit_button.disabled = True

send_button.on_click(on_send)
exit_button.on_click(on_exit)

print("=" * 50)
print("  EMOTION DETECTION CHATBOT")
print("=" * 50)
display(widgets.HBox([text_input, send_button, exit_button]), out)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

  EMOTION DETECTION CHATBOT


Output()